# Module 04 — Building Neural Networks: From Scratch to `nn.Module`

**Prerequisites:** Module 03 (Autograd and Computational Graphs)
**Time:** ~120 minutes

## Learning Objectives

- Implement a linear layer manually as `y = x @ W + b`, and train it using raw autograd.
- Explain what `nn.Linear` abstracts away, having just built it by hand.
- Build a multi-layer network using `nn.Module` and explain parameter registration.
- Compare `nn.Sequential` vs `nn.Module` subclass and know when to use each.
- Use `nn.ModuleList` and `nn.ModuleDict` for dynamic architectures.
- Explain the role of activation functions and compare ReLU, Sigmoid, Tanh, GELU, and LeakyReLU.
- Apply weight initialization strategies.
- Use forward hooks to inspect intermediate activations.


In [ ]:
import torch
import torch.nn as nn


## Step 1: A Linear Layer, By Hand

Nearly every neural network is built from layers that compute $y = xW + b$ (a **linear transformation**), followed by a nonlinear **activation function**. Let's implement the linear part manually, so `nn.Linear` never feels like a black box.

We'll build a toy regression problem: predict `y` from `x` where the true relationship is `y = 3x + 2` (plus noise), and see if gradient descent can recover something close to `weight=3, bias=2` using nothing but tensors and autograd.


In [ ]:
torch.manual_seed(0)

# Generate synthetic data: y = 3x + 2 + small noise
x = torch.linspace(-5, 5, 100).unsqueeze(1)          # shape (100, 1) -- 100 samples, 1 feature each
true_weight, true_bias = 3.0, 2.0
y = true_weight * x + true_bias + torch.randn_like(x) * 0.5   # add noise so it's not a perfectly clean line

print("x shape:", x.shape, " y shape:", y.shape)


We deliberately shaped `x` as `(100, 1)` rather than `(100,)` — a column of 100 single-feature samples — because that's the shape real feature data takes: `(num_samples, num_features)`. Get comfortable with this shape convention now; it appears everywhere from here on.


In [ ]:
# Manually create the parameters. requires_grad=True: we want gradients w.r.t. these.
W = torch.randn(1, 1, requires_grad=True)   # shape (in_features=1, out_features=1)
b = torch.zeros(1, requires_grad=True)

learning_rate = 0.01
n_steps = 200

for step in range(n_steps):
    # Forward pass: the manual version of what nn.Linear does internally
    y_pred = x @ W + b

    # Loss: mean squared error, computed manually
    loss = ((y_pred - y) ** 2).mean()

    # Backward pass: autograd computes d(loss)/dW and d(loss)/db
    loss.backward()

    # Parameter update: manual gradient descent step.
    # torch.no_grad() because this update itself should NOT be tracked by autograd.
    with torch.no_grad():
        W -= learning_rate * W.grad
        b -= learning_rate * b.grad

    # Reset gradients -- required because they accumulate by default (Module 03)
    W.grad.zero_()
    b.grad.zero_()

    if step % 40 == 0:
        print(f"step {step:3d} | loss {loss.item():.4f} | W {W.item():.3f} | b {b.item():.3f}")

print(f"\nFinal: W={W.item():.3f} (true={true_weight}), b={b.item():.3f} (true={true_bias})")


Notice the recovered `W` and `b` land close to the true values `3.0` and `2.0` — gradient descent, using nothing but the autograd machinery from the previous module, found the underlying relationship in the noisy data. This entire loop — forward pass, loss, `.backward()`, manual update, zero gradients — *is* training, in its most stripped-down form. Every abstraction from here forward is just a more convenient way to write exactly this.


## Step 2: The Same Thing, With `nn.Linear`

`nn.Linear(in_features, out_features)` does exactly what we just wrote by hand: it creates a weight matrix and bias vector (shapes `(out_features, in_features)` and `(out_features,)`), both with `requires_grad=True` already set, and computes `x @ weight.T + bias` when called.


In [ ]:
torch.manual_seed(0)

layer = nn.Linear(in_features=1, out_features=1)

print("weight shape:", layer.weight.shape)   # (1, 1)
print("bias shape:  ", layer.bias.shape)     # (1,)
print("weight requires_grad:", layer.weight.requires_grad)   # True automatically
print("weight type:", type(layer.weight))  # nn.Parameter, not just a tensor

# Using the layer is just calling it like a function:
sample_output = layer(x[:3])
print("\noutput for first 3 samples:", sample_output.squeeze())


`layer.weight` and `layer.bias` are `nn.Parameter` objects — tensors that are automatically registered so that `layer.parameters()` (used to hand parameters to an optimizer) finds them without you doing anything manual. That registration is the main thing `nn.Linear` and `nn.Module` add on top of what you built by hand above: **automatic bookkeeping of every parameter in the model**, so that "grab every trainable number and hand it to the optimizer" is one line of code instead of a manually maintained list.


## Step 3: Why We Need Activation Functions

A linear layer alone can only represent straight-line (technically: linear) relationships. Stacking multiple linear layers *without* anything nonlinear between them doesn't help — the composition of any number of linear functions is still just one linear function.


In [ ]:
# Two stacked linear layers with NO activation between them
stacked_linear = nn.Sequential(
    nn.Linear(1, 10),
    nn.Linear(10, 1),
)

# This is mathematically equivalent to a SINGLE linear layer -- verify by checking
# that the relationship between input and output is still a straight line.
test_x = torch.linspace(-5, 5, 5).unsqueeze(1)
test_y = stacked_linear(test_x).squeeze()
print("output:", test_y)
diffs = test_y[1:] - test_y[:-1]
print("differences between consecutive outputs (should be ~constant for a straight line):")
print(diffs)


The (nearly) constant differences confirm it: even with two layers and 10 "hidden" values in between, the overall input-output relationship is still just a straight line. Adding a **nonlinear activation function** between the layers is what allows the network to represent curves, and combinations of curves — which is what lets neural networks approximate essentially any function, given enough layers and width.


In [ ]:
model_with_activation = nn.Sequential(
    nn.Linear(1, 10),
    nn.ReLU(),        # the nonlinearity
    nn.Linear(10, 1),
)

test_y2 = model_with_activation(test_x).squeeze()
diffs2 = test_y2[1:] - test_y2[:-1]
print("differences between consecutive outputs (now NOT constant -- it's curved):")
print(diffs2)


### Activation Function Comparison

Here are the most common activation functions you'll encounter:

| Function | Formula | Range | When to use |
|---|---|---|---|
| **ReLU** | `max(0, x)` | [0, ∞) | Default hidden layer activation. Fast, simple, works well |
| **LeakyReLU** | `x if x>0 else 0.01x` | (-∞, ∞) | When you suspect "dead ReLU" neurons |
| **GELU** | `x * Φ(x)` | (-∞, ∞) | Transformers (BERT, GPT). Smooth version of ReLU |
| **Sigmoid** | `1/(1+e^(-x))` | (0, 1) | Binary output (probabilities). Avoid in hidden layers (vanishing gradient) |
| **Tanh** | `(e^x - e^(-x))/(e^x + e^(-x))` | (-1, 1) | RNNs, some hidden layers. Zero-centered |
| **Softmax** | `e^(x_i) / Σe^(x_j)` | (0, 1) | Multi-class output layer (never in hidden layers) |


In [ ]:
# Comparing activations on the same input
x = torch.linspace(-3, 3, 7)
print(f"{'x':>6} | {'ReLU':>8} | {'LeakyReLU':>10} | {'Sigmoid':>8} | {'Tanh':>8} | {'GELU':>8}")
print("-" * 65)

relu_out = torch.relu(x)
leaky_out = nn.LeakyReLU(0.01)(x)
sigmoid_out = torch.sigmoid(x)
tanh_out = torch.tanh(x)
gelu_out = nn.GELU()(x)

for i in range(len(x)):
    print(f"{x[i].item():6.2f} | {relu_out[i].item():8.4f} | {leaky_out[i].item():10.4f} | "
          f"{sigmoid_out[i].item():8.4f} | {tanh_out[i].item():8.4f} | {gelu_out[i].item():8.4f}")


### The "Dead ReLU" Problem

If a ReLU neuron always receives negative inputs, its output is always 0, and its gradient is always 0 — so it never updates. This neuron is "dead." LeakyReLU avoids this by allowing a small gradient for negative inputs.


In [ ]:
# Demonstration: ReLU kills all negative values
x = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0], requires_grad=True)
y = torch.relu(x)
y.sum().backward()

print("input:   ", x.data)
print("ReLU out:", y.data)
print("gradient:", x.grad)   # 0 for all negative inputs — these neurons are "dead"


## Step 4: Building a Model as an `nn.Module` Subclass

For anything beyond the simplest cases, you'll define your model as a Python class inheriting from `nn.Module`. This is the pattern you'll use for the rest of the course.


In [ ]:
class SimpleRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()   # MUST be called first -- see Module 00's warning about this
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # forward() defines what happens when you call model(x).
        # You never call forward() directly -- calling the model itself
        # (model(x)) triggers some internal nn.Module bookkeeping, then calls forward() for you.
        x = self.layer1(x)
        x = self.activation(x)
        x = self.layer2(x)
        return x


model = SimpleRegressor(input_dim=1, hidden_dim=16, output_dim=1)
print(model)


**Why does assigning `self.layer1 = nn.Linear(...)` inside `__init__` matter?** Because `nn.Module.__init__()` (called via `super().__init__()`) sets up an internal registry. Whenever you assign an `nn.Module` (like `nn.Linear`) as an attribute of `self`, it gets automatically added to that registry. This is *why* `model.parameters()` below finds all the weights and biases, recursively, from every sub-layer, without you doing anything manual.


In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total trainable parameters: {total_params}")

for name, param in model.named_parameters():
    print(f"{name:20s} shape={tuple(param.shape)}  numel={param.numel()}")


## `nn.Sequential` vs `nn.Module` Subclass

**`nn.Sequential`** is a shortcut for simple chains of layers (data flows through each layer in order, no branching). **`nn.Module` subclass** is for anything more complex.


In [ ]:
# These two models are functionally identical:

# Option 1: Sequential (quick and clean for simple chains)
model_seq = nn.Sequential(
    nn.Linear(1, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)

# Option 2: Module subclass (more flexible)
class ModelClass(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(1, 16)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(16, 1)
    
    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

model_cls = ModelClass()

print("Sequential params:", sum(p.numel() for p in model_seq.parameters()))
print("Module subclass params:", sum(p.numel() for p in model_cls.parameters()))


**When to use which:**

| Use `nn.Sequential` when... | Use `nn.Module` subclass when... |
|---|---|
| Data flows straight through in order | You need branching, skip connections, or conditionals |
| No intermediate values needed | You want to inspect/return intermediate activations |
| Quick prototyping | Production models, anything non-trivial |

In practice, most real models use `nn.Module` subclasses with `nn.Sequential` for sub-components.


## `nn.ModuleList` and `nn.ModuleDict`

If you store modules in a plain Python `list` or `dict`, their parameters **won't be registered** — `model.parameters()` won't find them. Use `nn.ModuleList` or `nn.ModuleDict` instead.


In [ ]:
# ❌ WRONG: plain Python list — parameters NOT registered
class BrokenDynamicModel(nn.Module):
    def __init__(self, n_layers):
        super().__init__()
        self.layers = [nn.Linear(10, 10) for _ in range(n_layers)]  # plain list!
    
    def forward(self, x):
        for layer in self.layers:
            x = torch.relu(layer(x))
        return x

broken = BrokenDynamicModel(3)
print("Broken model params:", sum(p.numel() for p in broken.parameters()))  # 0! Nothing registered!

# ✅ CORRECT: nn.ModuleList — parameters ARE registered
class CorrectDynamicModel(nn.Module):
    def __init__(self, n_layers):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(10, 10) for _ in range(n_layers)])
    
    def forward(self, x):
        for layer in self.layers:
            x = torch.relu(layer(x))
        return x

correct = CorrectDynamicModel(3)
print("Correct model params:", sum(p.numel() for p in correct.parameters()))  # 330


In [ ]:
# nn.ModuleDict: register modules by name
class MultiHeadModel(nn.Module):
    def __init__(self, input_dim, task_dims):
        super().__init__()
        self.shared = nn.Linear(input_dim, 32)
        self.heads = nn.ModuleDict({
            name: nn.Linear(32, dim) for name, dim in task_dims.items()
        })
    
    def forward(self, x, task_name):
        shared_out = torch.relu(self.shared(x))
        return self.heads[task_name](shared_out)

model = MultiHeadModel(10, {"classify": 5, "regress": 1})
print(model)
print(f"\nTotal params: {sum(p.numel() for p in model.parameters())}")


## Custom `nn.Parameter`

Sometimes you want a manually created tensor to be treated as a trainable parameter. Wrapping it in `nn.Parameter` makes it appear in `model.parameters()`.


In [ ]:
class ScaledModel(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)
        # A learnable scaling factor — starts at 1.0, trained alongside everything else
        self.scale = nn.Parameter(torch.tensor(1.0))
    
    def forward(self, x):
        return self.scale * self.linear(x)

model = ScaledModel(5, 1)
for name, param in model.named_parameters():
    print(f"{name:15s} shape={tuple(param.shape)}")


## Weight Initialization

How weights start matters more than you'd expect. Bad initialization can lead to vanishing or exploding activations before training even begins.


In [ ]:
layer = nn.Linear(100, 100)
print("default init — weight stats:")
print(f"  mean={layer.weight.data.mean():.4f}, std={layer.weight.data.std():.4f}")

# Xavier/Glorot: good for sigmoid/tanh activations
nn.init.xavier_uniform_(layer.weight)
print("\nxavier_uniform — weight stats:")
print(f"  mean={layer.weight.data.mean():.4f}, std={layer.weight.data.std():.4f}")

# Kaiming/He: good for ReLU activations (the most common choice)
nn.init.kaiming_normal_(layer.weight, mode='fan_in', nonlinearity='relu')
print("\nkaiming_normal (for ReLU) — weight stats:")
print(f"  mean={layer.weight.data.mean():.4f}, std={layer.weight.data.std():.4f}")

# Zero-init (for bias, or specific architectural choices)
nn.init.zeros_(layer.bias)
print("\nbias after zeros_:", layer.bias.data[:5])


In [ ]:
# Apply initialization to an entire model
def init_weights(module):
    if isinstance(module, nn.Linear):
        nn.init.kaiming_normal_(module.weight, nonlinearity='relu')
        if module.bias is not None:
            nn.init.zeros_(module.bias)

model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)

model.apply(init_weights)   # recursively applies to every sub-module
print("Initialization applied to all layers.")
for name, param in model.named_parameters():
    if 'weight' in name:
        print(f"  {name}: mean={param.data.mean():.4f}, std={param.data.std():.4f}")


## `nn.Flatten`

Convolutional layers output tensors with shape `(batch, channels, height, width)`, but linear layers expect `(batch, features)`. `nn.Flatten` bridges the gap by collapsing all non-batch dimensions into one.


In [ ]:
# Simulating a CNN -> Linear transition
conv_output = torch.randn(8, 16, 4, 4)   # batch=8, channels=16, height=4, width=4
print("before flatten:", conv_output.shape)

flatten = nn.Flatten()   # flattens all dims except batch (dim 0)
flat = flatten(conv_output)
print("after flatten:", flat.shape)   # (8, 16*4*4) = (8, 256)

# Now a linear layer can process it
linear = nn.Linear(256, 10)
output = linear(flat)
print("after linear:", output.shape)   # (8, 10)


## Shape Tracking Through the Model

Let's explicitly trace how the shape changes as data flows through a model — a habit worth building now, since it becomes essential once models get deeper.

```
Input
  |  shape (batch, 1)
  v
nn.Linear(1, 16)
  |  shape (batch, 16)
  v
nn.ReLU()
  |  shape (batch, 16)  -- ReLU doesn't change shape, only values
  v
nn.Linear(16, 1)
  |  shape (batch, 1)
  v
Output
```


In [ ]:
model = SimpleRegressor(input_dim=1, hidden_dim=16, output_dim=1)
batch = x[:8]   # take 8 samples, shape (8, 1)
print("input shape: ", batch.shape)

out1 = model.layer1(batch)
print("after layer1:", out1.shape)

out2 = model.activation(out1)
print("after ReLU:  ", out2.shape)   # unchanged from out1

out3 = model.layer2(out2)
print("after layer2:", out3.shape)

# Sanity check: this should match calling the model directly
direct = model(batch)
print("\nmatches model(batch)?", torch.allclose(out3, direct))


## Forward Hooks (Preview)

Hooks let you inspect (or modify) a layer's input/output without changing the model code. Useful for debugging, visualization, and feature extraction.


In [ ]:
# Register a hook that prints the shape of layer1's output
activations = {}

def save_activation(name):
    def hook(module, input, output):
        activations[name] = output.detach()
        print(f"  hook: {name} output shape = {output.shape}")
    return hook

# Attach hooks
handle1 = model.layer1.register_forward_hook(save_activation('layer1'))
handle2 = model.layer2.register_forward_hook(save_activation('layer2'))

print("Running forward pass with hooks:")
output = model(batch)

print("\nCaptured activations:")
for name, act in activations.items():
    print(f"  {name}: shape={act.shape}, mean={act.mean():.4f}")

# Clean up: remove hooks when done
handle1.remove()
handle2.remove()


## Model Summary Utility

A handy helper to quickly see all layers, their parameter counts, and total model size:


In [ ]:
def model_summary(model):
    """Print a summary of model layers and parameter counts."""
    print(f"{'Layer':30s} {'Shape':20s} {'Params':>10s}")
    print("-" * 62)
    total = 0
    for name, param in model.named_parameters():
        count = param.numel()
        total += count
        print(f"{name:30s} {str(tuple(param.shape)):20s} {count:10,d}")
    print("-" * 62)
    print(f"{'Total':30s} {'':20s} {total:10,d}")
    print(f"Model size: {total * 4 / 1024:.1f} KB (float32)")

# Test on a bigger model
big_model = nn.Sequential(
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)

model_summary(big_model)


## 🐛 Debugging Challenge

The model below fails when given a batch of data. Find the bug before reading on.


In [ ]:
class BrokenModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(1, 8)
        self.layer2 = nn.Linear(4, 1)   # bug is here

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        return x

broken_model = BrokenModel()
try:
    output = broken_model(x[:4])
except RuntimeError as e:
    print("Error:", e)


**Diagnosis:** `layer1` outputs 8 features (`nn.Linear(1, 8)`), but `layer2` expects an *input* of 4 features (`nn.Linear(4, 1)`). Every layer's output size must match the next layer's expected input size.

**Fix:** `layer2` should be `nn.Linear(8, 1)` to match `layer1`'s output.


## Building a Configurable MLP

Let's combine several concepts (ModuleList, activation choice, initialization) into a reusable MLP builder:


In [ ]:
class MLP(nn.Module):
    """A configurable multi-layer perceptron."""
    def __init__(self, layer_sizes, activation='relu', dropout=0.0):
        super().__init__()
        
        # Choose activation
        activations = {'relu': nn.ReLU, 'gelu': nn.GELU, 'tanh': nn.Tanh, 'leaky_relu': nn.LeakyReLU}
        act_fn = activations[activation]
        
        # Build layers dynamically
        layers = []
        for i in range(len(layer_sizes) - 1):
            layers.append(nn.Linear(layer_sizes[i], layer_sizes[i+1]))
            if i < len(layer_sizes) - 2:   # no activation/dropout after the last layer
                layers.append(act_fn())
                if dropout > 0:
                    layers.append(nn.Dropout(dropout))
        
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

# Usage examples:
mlp_small = MLP([784, 128, 10], activation='relu')
mlp_large = MLP([784, 512, 256, 128, 10], activation='gelu', dropout=0.1)

print("Small MLP:")
print(mlp_small)
print(f"Params: {sum(p.numel() for p in mlp_small.parameters()):,}")

print(f"\nLarge MLP params: {sum(p.numel() for p in mlp_large.parameters()):,}")


## Exercises

🟢 **Beginner 1:** Build an `nn.Sequential` model with `nn.Linear(1, 5)`, `nn.ReLU()`, `nn.Linear(5, 1)`. Print `model` and count total parameters.

🟢 **Beginner 2:** Create an `nn.Linear(10, 5)` layer. Print the weight and bias shapes. Verify that `weight.requires_grad` is `True`.

🟡 **Intermediate 1:** Build an `nn.Module` subclass called `TwoHiddenLayerNet` with hidden layers of width 32 and 16 (input/output dim as constructor args), each followed by `nn.ReLU()`. Print total trainable parameters.

🟡 **Intermediate 2:** Use the `MLP` class above to create a model with layers `[100, 64, 32, 10]`. Run a random batch of shape `(8, 100)` through it and verify the output shape is `(8, 10)`.

🔴 **Challenge 1:** Build a model that stores 5 linear layers in an `nn.ModuleList`, then in `forward()` applies each layer with ReLU in a loop. Verify that all parameters are registered.

🔴 **Challenge 2:** Retrain the manual `W`/`b` linear regression from Step 1, but this time use `nn.Linear(1, 1)` and its `.parameters()` inside the training loop, updating parameters manually with `torch.no_grad()` and `param.grad`.


In [ ]:
# Space for your exercise solutions



## Common Mistakes

- **Forgetting `super().__init__()`** — as covered in Module 00, this breaks `nn.Module`'s internal parameter registration.
- **Mismatched layer dimensions** — one layer's output size must equal the next layer's input size; PyTorch will not silently reshape this for you.
- **Using a plain Python list instead of `nn.ModuleList`** — parameters won't be registered, and the optimizer won't update them.
- **Stacking linear layers with no activation between them** — mathematically collapses to a single linear layer, wasting the extra parameters and depth.
- **Calling `model.forward(x)` directly instead of `model(x)`** — this skips internal `nn.Module` bookkeeping (hooks, mode-switching) and should be avoided.
- **Forgetting to match Kaiming init with ReLU and Xavier init with sigmoid/tanh** — wrong init for the activation can lead to vanishing/exploding activations.

## Mental Model

An `nn.Module` is a *box that knows what's inside it*. When you put `nn.Linear` layers (or other modules) inside as attributes, the outer box automatically knows to include their parameters when someone asks "what are all your trainable numbers?" (`model.parameters()`). `forward()` is simply "what do you do when someone hands you an input" — the wiring diagram of how data flows through everything the box contains.

## Key Takeaways

- `y = x @ W + b`, computed and trained manually with autograd, is exactly what `nn.Linear` does — the abstraction adds parameter registration and convenience, not new math.
- Nonlinear activation functions (ReLU, GELU, Tanh, etc.) are what let stacked layers represent curved, complex functions.
- `nn.Module` subclasses register any `nn.Module`/`nn.Parameter` assigned as `self.<name>`, which is why `model.parameters()` finds everything.
- Use `nn.ModuleList`/`nn.ModuleDict` instead of plain Python lists/dicts to hold dynamic collections of layers.
- Weight initialization matters: Kaiming for ReLU, Xavier for sigmoid/tanh.
- Always track shapes as data flows through a model — mismatched layer dimensions are one of the most common errors.

## What's Next

**Module 05 — Loss Functions, Optimizers, and the Training Loop** replaces the manual gradient-descent update step you wrote by hand in this notebook with PyTorch's `torch.optim` optimizers, and introduces the standard loss functions used for classification and regression.

## Checklist

- [ ] I implemented a linear layer manually and trained it with raw autograd
- [ ] I can explain what `nn.Linear` and `nn.Module` add on top of raw tensors + autograd
- [ ] I know when to use `nn.Sequential` vs an `nn.Module` subclass
- [ ] I understand why `nn.ModuleList` is needed instead of a plain Python list
- [ ] I can explain why activation functions are necessary between linear layers
- [ ] I can compare ReLU, LeakyReLU, GELU, Sigmoid, and Tanh and know when to use each
- [ ] I can apply weight initialization to a model
- [ ] I can trace tensor shapes through a multi-layer model and diagnose a shape mismatch
